In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import skfuzzy as fuzz
import matplotlib.pyplot as plt
import seaborn as sns
from skfuzzy import control as ctrl
from ipywidgets import interact, FloatSlider, Dropdown
from LQ.DANN import calculate_points, apply_fuzzy_correction
from IPython.display import HTML, display
import os

subjects = ['math', 'physics', 'literature', 'biology', 'chemistry']
if not os.path.exists("Dena.csv"):
    print("Файл данных не найден! Сначала выполните генерацию данных.")
    exit()

css = """
<style>
body {
    background-color: #e9ecef !important;
    color: #343a40 !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}
.widget-box {
    background: #f8f9fa;
    padding: 15px;
    border-radius: 10px;
    margin-bottom: 15px;
    border: 1px solid #dee2e6;
}
.widget-label {
    font-weight: 600 !important;
    color: #212529 !important;
    font-size: 16px !important;
    margin-bottom: 5px !important;
    display: block;
}
.widget-slider {
    width: 95% !important;
    margin: 10px 0 !important;
}
.widget-readout {
    font-weight: 600 !important;
    font-size: 18px !important;
    color: #0d6efd !important;
    background: #e9ecef;
    padding: 5px 10px;
    border-radius: 4px;
    display: inline-block;
    margin-top: 5px;
}
.recommendation-box {
    padding: 25px;
    margin: 25px 0;
    border-radius: 12px;
    background: #f8f9fa;
    border: 1px solid #ced4da;
    box-shadow: 0 4px 6px rgba(0,0,0,0.05);
}
.recommendation-text {
    color: #0d6efd;
    font-size: 26px !important;
    font-weight: 700;
    margin-bottom: 20px;
    text-align: center;
    padding: 15px;
    border-radius: 8px;
    background: #e2e8f0;
}
.detail-container {
    background: #f1f3f5;
    padding: 20px;
    border-radius: 10px;
}
.detail-header {
    font-size: 20px;
    font-weight: 600;
    color: #212529;
    margin-bottom: 15px;
    padding-bottom: 10px;
    border-bottom: 1px solid #ced4da;
}
.detail-item {
    margin: 12px 0;
    padding: 12px 15px;
    background: #ffffff;
    border-radius: 6px;
    font-size: 18px;
    box-shadow: 0 1px 2px rgba(0,0,0,0.05);
    border-left: 3px solid #0d6efd;
}
.detail-label {
    font-weight: 600;
    color: #495057;
    display: inline-block;
    min-width: 180px;
}
.detail-value {
    font-weight: 700;
    color: #0d6efd;
}
.detail-description {
    color: #6c757d;
    font-size: 16px;
    margin-top: 5px;
    font-style: normal;
}
.container {
    max-width: 900px;
    margin: 0 auto;
    padding: 20px;
    background: #f8f9fa;
    border-radius: 15px;
    box-shadow: 0 5px 15px rgba(0,0,0,0.08);
}
</style>
"""
display(HTML(css))

interest_mapping = {
    'very_low': 1,
    'low': 2,
    'medium': 3,
    'high': 4,
    'very_high': 5
}

interest_levels = {
    1: 'очень низкий',
    2: 'низкий',
    3: 'средний',
    4: 'высокий',
    5: 'очень высокий'
}

stress_levels = {
    1: 'очень низкий',
    2: 'низкий',
    3: 'средний',
    4: 'высокий',
    5: 'очень высокий'
}

def get_interest_label(value):
    if isinstance(value, str):
        num_value = interest_mapping.get(value.lower(), 3)
    else:
        num_value = round(float(value))
    return interest_levels.get(num_value, 'не определен')

def get_stress_label(value):
    try:
        num_value = float(value)
        if 1.0 <= num_value < 2.0:
            return "очень низкий"
        elif 2.0 <= num_value < 3.0:
            return "низкий"
        elif 3.0 <= num_value < 4.0:
            return "средний"
        elif 4.0 <= num_value < 5.0:
            return "высокий"
        elif num_value >= 5.0:
            return "очень высокий"
        else:
            return "не определен"
    except:
        return "не определен"

try:
    df = pd.read_csv("Dena.csv")
    print("Данные успешно загружены!")
    print(f"Записей: {len(df)}")
except Exception as e:
    print(f"Ошибка загрузки данных: {str(e)}")
    df = pd.DataFrame()

def get_recommendation(student_data):
    try:
        required = ['attendance', 'interest', 'stress_level', 'study_hours'] + [f'grade_{s}' for s in subjects]
        for field in required:
            if field not in student_data:
                raise ValueError(f"Отсутствует поле: {field}")

        if not (1 <= student_data['interest'] <= 5):
            raise ValueError("Уровень интереса должен быть между 1 и 5")

        attendance = student_data['attendance']
        grades = [student_data[f'grade_{subj}'] for subj in subjects]
        academic_val = np.mean(grades)
        min_grade_val = min(grades) if grades else 70.0

        base_points = calculate_points({
            'academic': academic_val,
            'interest': student_data['interest'],
            'stress': student_data['stress_level'],
            'time': student_data['study_hours']
        })

        corrected_points, advice, correction_value = apply_fuzzy_correction(
            base_points,
            student_data,
            subjects
        )

        if corrected_points >= 9.0:
            recommendation = "Экспертная программа+"
            level = 7
        elif corrected_points >= 8.0:
            recommendation = "Экспертная программа"
            level = 6
        elif corrected_points >= 6.5:
            recommendation = "Продвинутый курс"
            level = 5
        elif corrected_points >= 5.0:
            recommendation = "Интенсивный курс"
            level = 4
        elif corrected_points >= 3.5:
            recommendation = "Базовый курс+"
            level = 3
        elif corrected_points >= 2.0:
            recommendation = "Базовый курс"
            level = 2
        else:
            recommendation = "Индивидуальная поддержка"
            level = 1

        academic_impact = round(academic_val/10 * 0.6, 1)
        interest_impact = round((student_data['interest']-1)*2.5 * 0.25, 1)
        time_impact = round(student_data['study_hours']/4 * 0.15, 1)
        stress_penalty = round(student_data['stress_level']*0.8, 1)

        return {
            'recommendation': recommendation,
            'level': level,
            'points': round(corrected_points, 1),
            'base_points': round(base_points, 1),
            'correction': correction_value,
            'advice': advice,
            'min_grade': min_grade_val,
            'grade_variance': np.var(grades) if grades else 0.0,
            'details': {
                'academic_impact': academic_impact,
                'interest_impact': interest_impact,
                'time_impact': time_impact,
                'stress_penalty': stress_penalty,
                'stress_label': get_stress_label(student_data['stress_level']),
                'stress_value': student_data['stress_level'],
                'min_grade': min_grade_val
            }
        }

    except Exception as e:
        print(f"Ошибка в формировании рекомендации: {str(e)}")
        return {
            'recommendation': "Базовый курс",
            'level': 2,
            'points': 5.0,
            'base_points': 5.0,
            'correction': 0.0,
            'advice': ["Не удалось сформировать рекомендацию"],
            'min_grade': 0,
            'grade_variance': 0.0,
            'details': {}
        }

def plot_recommendation_levels():
    levels = [
        (0.0, 2.0, "Индивидуальная поддержка", "#FF6B6B"),
        (2.0, 3.5, "Базовый курс", "#FFD93D"),
        (3.5, 5.0, "Базовый курс+", "#FFE66D"),
        (5.0, 6.5, "Интенсивный курс", "#6ECB63"),
        (6.5, 8.0, "Продвинутый курс", "#4CAF50"),
        (8.0, 9.0, "Экспертная программа", "#2E7D32"),
        (9.0, 10.0, "Экспертная программа+", "#1B5E20")
    ]

    plt.figure(figsize=(12, 3))
    for start, end, label, color in levels:
        plt.fill_between([start, end], [0, 0], [1, 1], color=color, alpha=0.6, label=label)

    plt.title("Уровни рекомендаций")
    plt.xlabel("Накопленные очки")
    plt.yticks([])
    plt.xlim(0, 10)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    plt.rcParams.update({
        'axes.edgecolor': '#333333',
        'axes.labelcolor': '#333333',
        'text.color': '#333333',
        'xtick.color': '#333333',
        'ytick.color': '#333333'
    })

    plt.show()

@interact(
    grade_math=FloatSlider(min=20, max=100, step=1, value=65, description="Математика"),
    grade_physics=FloatSlider(min=20, max=100, step=1, value=60, description="Физика"),
    grade_literature=FloatSlider(min=20, max=100, step=1, value=75, description="Литература"),
    grade_biology=FloatSlider(min=20, max=100, step=1, value=70, description="Биология"),
    grade_chemistry=FloatSlider(min=20, max=100, step=1, value=68, description="Химия"),
    interest=FloatSlider(min=1, max=5, step=0.1, value=3.0, description="Интерес (1-5)"),
    stress_level=FloatSlider(min=1, max=5, step=0.1, value=2.5, description="Стресс (1-5)"),
    study_hours=FloatSlider(min=4, max=40, step=1, value=25, description="Время учебы (ч/нед)"),
    attendance=FloatSlider(min=0, max=100, step=1, value=80, description="Посещаемость (%)")
)
def show_recommendation(grade_math, grade_physics, grade_literature,
                       grade_biology, grade_chemistry,
                       interest, stress_level, study_hours, attendance):
    student_data = {
        'grade_math': grade_math,
        'grade_physics': grade_physics,
        'grade_literature': grade_literature,
        'grade_biology': grade_biology,
        'grade_chemistry': grade_chemistry,
        'interest': interest,
        'stress_level': stress_level,
        'study_hours': study_hours,
        'attendance': attendance
    }

    result = get_recommendation(student_data)
    advice_html = "<br>".join(result['advice']) if result['advice'] else "Нет специальных рекомендаций"

    display(HTML(f"""
<div class='recommendation-box'>
    <div class='recommendation-text'>📚 {result['recommendation']} (Уровень {result['level']}/7)</div>

    <div class='detail-container'>
        <div class='detail-header'>🔍 Детализация расчета</div>

        <div class='detail-item'>
            <span class='detail-label'>Базовые баллы:</span>
            <span class='detail-value'>{result['base_points']}/10</span>
            <div class='detail-description'>Рассчитываются на основе средней оценки, интереса и учебного времени</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Нечеткая коррекция:</span>
            <span class='detail-value'>{result['correction']:.1f}</span>
            <div class='detail-description'>Корректировка на основе стресса, разброса оценок и других факторов</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Итоговые баллы:</span>
            <span class='detail-value'>{result['points']}/10</span>
            <div class='detail-description'>Базовые баллы + коррекция, определяют уровень программы</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Минимальная оценка:</span>
            <span class='detail-value'>{result['min_grade']:.0f}</span>
            <div class='detail-description'>Самая низкая оценка среди всех предметов</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Разброс оценок:</span>
            <span class='detail-value'>{result['grade_variance']:.1f}</span>
            <div class='detail-description">Показывает неравномерность успеваемости (чем выше, тем больше разница между предметами)</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Персональные рекомендации:</span>
            <div class='detail-advice'>{advice_html}</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Вклад успеваемости:</span>
            <span class='detail-value'>+{result['details']['academic_impact']:.1f}</span>
            <div class='detail-description'>(60% от средней оценки: {np.mean(list(student_data.values())[:5]):.1f})</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Вклад интереса:</span>
            <span class='detail-value'>+{result['details']['interest_impact']:.1f}</span>
            <div class='detail-description'>(25% от уровня интереса: {get_interest_label(student_data['interest'])})</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Вклад учебного времени:</span>
            <span class='detail-value'>+{result['details']['time_impact']:.1f}</span>
            <div class='detail-description'>(15% от учебных часов: {student_data['study_hours']} ч/нед)</div>
        </div>

        <div class='detail-item'>
            <span class='detail-label'>Штраф за стресс:</span>
            <span class='detail-value'>-{result['details']['stress_penalty']:.1f}</span>
            <div class='detail-description'>({result['details']['stress_label']}, снижает общий балл)</div>
        </div>
    </div>
</div>
"""))

    plot_recommendation_levels()

    plt.figure(figsize=(10, 6))
    subjects_grades = [
        student_data['grade_math'],
        student_data['grade_physics'],
        student_data['grade_literature'],
        student_data['grade_biology'],
        student_data['grade_chemistry']
    ]
    sns.barplot(x=['Математика', 'Физика', 'Литература', 'Биология', 'Химия'],
            y=subjects_grades, palette="viridis")
    plt.axhline(y=50, color='r', linestyle='--', label='Минимальный порог')
    plt.title("Оценки по предметам")
    plt.ylabel("Оценка")
    plt.ylim(0, 100)
    plt.legend()
    plt.show()

print("\nТестирование на реальных данных:")
try:
    demo_df = pd.read_csv("Dena.csv").sample(3)
    for idx, row in demo_df.iterrows():
        student_data = {
            'attendance': row['attendance'],
            'interest': row['interest'],
            'stress_level': row['stress_level'],
            'study_hours': row['study_hours'],
            'grade_math': row['grade_math'],
            'grade_physics': row['grade_physics'],
            'grade_literature': row['grade_literature'],
            'grade_biology': row['grade_biology'],
            'grade_chemistry': row['grade_chemistry']
        }

        recommendation = get_recommendation(student_data)
        avg_grade = np.mean([row[f'grade_{subj}'] for subj in subjects])

        print(f"\nСтудент #{idx+1}")
        print(f"Очки: {recommendation['points']:.1f}/10")
        print(f"Рекомендация: {recommendation['recommendation']} (Уровень {recommendation['level']})")
        print(f"Минимальная оценка: {recommendation['min_grade']:.0f}")
        print(f"Разброс оценок: {recommendation['grade_variance']:.1f}")
        print("Детали:")
        print(f"  - Успеваемость: {avg_grade:.1f}")
        print(f"  - Интерес: {row['interest']} ({get_interest_label(row['interest'])})")
        print(f"  - Стресс: {row['stress_level']} ({get_stress_label(row['stress_level'])})")
        print(f"  - Время учебы: {row['study_hours']} ч/нед")
        print(f"  - Посещаемость: {row['attendance']}%")
        print("Рекомендации:")
        for advice in recommendation['advice']:
            print(f"  - {advice}")
        print("-"*50)

except Exception as e:
    print(f"Ошибка тестирования: {str(e)}")

Данные успешно загружены!
Записей: 80000


interactive(children=(FloatSlider(value=65.0, description='Математика', min=20.0, step=1.0), FloatSlider(value…


Тестирование на реальных данных:

Студент #24104
Очки: 0.0/10
Рекомендация: Индивидуальная поддержка (Уровень 1)
Минимальная оценка: 20
Разброс оценок: 16.6
Детали:
  - Успеваемость: 24.8
  - Интерес: 3 (средний)
  - Стресс: 4 (высокий)
  - Время учебы: 9 ч/нед
  - Посещаемость: 75.0%
Рекомендации:
  - Требуется консультация препода
  - Срочно улучшить оценки по: math (30), physics (29), literature (21), biology (24), chemistry (20)
  - Увеличить учебные часы для улучшения результатов
--------------------------------------------------

Студент #15448
Очки: 3.6/10
Рекомендация: Базовый курс+ (Уровень 3)
Минимальная оценка: 20
Разброс оценок: 172.4
Детали:
  - Успеваемость: 36.0
  - Интерес: 5 (очень высокий)
  - Стресс: 2 (низкий)
  - Время учебы: 16 ч/нед
  - Посещаемость: 99.0%
Рекомендации:
  - Срочно улучшить оценки по: literature (26), biology (31), chemistry (20)
  - Сбалансировать усилия по всем предметам
--------------------------------------------------

Студент #2215
Очки: 0.